# Benchmarks y Comparación de Protocolos — Práctica

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/18_intro_a_apis/code/05_benchmarks.ipynb)

En este notebook mediremos la latencia y el tamaño de payload de distintos
paradigmas de comunicación para entender sus diferencias de rendimiento.

## 1. Setup

In [ ]:
%pip install requests httpx websockets matplotlib numpy

In [ ]:
import requests
import time
import numpy as np
import matplotlib.pyplot as plt
import json
import asyncio
import websockets

### Funciones auxiliares

In [ ]:
def stats(tiempos, label):
    """Imprime estadísticas de latencia."""
    arr = np.array(tiempos) * 1000  # a milisegundos
    print(f"\n--- {label} ---")
    print(f"  N:      {len(arr)}")
    print(f"  Media:  {arr.mean():.2f} ms")
    print(f"  Mediana:{np.median(arr):.2f} ms")
    print(f"  P95:    {np.percentile(arr, 95):.2f} ms")
    print(f"  Min:    {arr.min():.2f} ms")
    print(f"  Max:    {arr.max():.2f} ms")
    return arr

## 2. Benchmark: REST latencia (sin sesión)

Realizamos 50 peticiones GET secuenciales a `httpbin.org/get`.
Cada petición crea una **nueva conexión TCP** (y potencialmente TLS).

In [ ]:
N = 50
URL = "https://httpbin.org/get"

tiempos_sin_sesion = []

print(f"Enviando {N} peticiones GET sin sesión...")
for i in range(N):
    t0 = time.perf_counter()
    r = requests.get(URL)
    t1 = time.perf_counter()
    tiempos_sin_sesion.append(t1 - t0)
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{N} completadas")

arr_sin_sesion = stats(tiempos_sin_sesion, "REST sin sesión")

## 3. Benchmark: REST con conexión persistente

Usamos `requests.Session()` para reutilizar la conexión TCP/TLS.
Esto evita el handshake en cada petición.

In [ ]:
tiempos_con_sesion = []

print(f"Enviando {N} peticiones GET con sesión...")
with requests.Session() as session:
    for i in range(N):
        t0 = time.perf_counter()
        r = session.get(URL)
        t1 = time.perf_counter()
        tiempos_con_sesion.append(t1 - t0)
        if (i + 1) % 10 == 0:
            print(f"  {i + 1}/{N} completadas")

arr_con_sesion = stats(tiempos_con_sesion, "REST con sesión")

mejora = (1 - arr_con_sesion.mean() / arr_sin_sesion.mean()) * 100
print(f"\nMejora con sesión: {mejora:.1f}%")

### ¿Por qué la sesión es más rápida?

Sin sesión, cada petición realiza:
1. **DNS lookup** — resolver el dominio
2. **TCP handshake** — 3-way handshake (SYN, SYN-ACK, ACK)
3. **TLS handshake** — negociación de cifrado (varios round-trips)
4. **HTTP request/response**

Con sesión, los pasos 1-3 solo ocurren en la primera petición.
Las siguientes reutilizan la conexión existente (HTTP keep-alive).

## 4. Benchmark: HTTP streaming

Comparamos la descarga de datos con y sin streaming.
Con streaming, procesamos los datos conforme llegan en lugar de esperar la respuesta completa.

In [ ]:
N_STREAM = 20  # menos iteraciones porque streaming es más lento

# Sin streaming: respuesta completa
tiempos_no_stream = []
print(f"Midiendo {N_STREAM} peticiones sin streaming...")
with requests.Session() as session:
    for i in range(N_STREAM):
        t0 = time.perf_counter()
        r = session.get("https://httpbin.org/get")
        _ = r.content  # forzar lectura completa
        t1 = time.perf_counter()
        tiempos_no_stream.append(t1 - t0)

arr_no_stream = stats(tiempos_no_stream, "Sin streaming")

# Con streaming: httpbin.org/stream/N devuelve N líneas JSON
tiempos_stream = []
print(f"\nMidiendo {N_STREAM} peticiones con streaming (10 líneas)...")
with requests.Session() as session:
    for i in range(N_STREAM):
        t0 = time.perf_counter()
        r = session.get("https://httpbin.org/stream/10", stream=True)
        lineas = []
        for linea in r.iter_lines():
            lineas.append(linea)
        t1 = time.perf_counter()
        tiempos_stream.append(t1 - t0)

arr_stream = stats(tiempos_stream, "Con streaming (10 líneas)")

print(f"\nNota: el endpoint de streaming devuelve más datos (10 objetos JSON),")
print(f"por lo que la comparación no es 1:1 en contenido.")
print(f"Lo importante es observar cómo el primer byte llega antes con streaming.")

### Tiempo al primer byte (TTFB) con streaming

In [ ]:
# Medir TTFB: tiempo hasta recibir el primer fragmento
ttfb_list = []
total_list = []

print("Midiendo TTFB vs tiempo total con streaming...")
with requests.Session() as session:
    for i in range(N_STREAM):
        t0 = time.perf_counter()
        r = session.get("https://httpbin.org/stream/10", stream=True)
        # TTFB: tiempo hasta que llega el primer chunk
        first = True
        for chunk in r.iter_lines():
            if first:
                ttfb = time.perf_counter() - t0
                ttfb_list.append(ttfb)
                first = False
        total = time.perf_counter() - t0
        total_list.append(total)

ttfb_arr = np.array(ttfb_list) * 1000
total_arr = np.array(total_list) * 1000

print(f"\nTTFB medio:        {ttfb_arr.mean():.2f} ms")
print(f"Tiempo total medio: {total_arr.mean():.2f} ms")
print(f"\nCon streaming podemos empezar a procesar datos")
print(f"{((1 - ttfb_arr.mean()/total_arr.mean())*100):.0f}% antes de que termine la transferencia.")

## 5. Benchmark: WebSocket latencia

Conectamos a un servidor de echo por WebSocket y medimos el round-trip time (RTT)
de 50 mensajes. La conexión se mantiene abierta durante toda la prueba.

In [ ]:
async def benchmark_websocket(n=50):
    """Mide RTT de n mensajes por WebSocket."""
    uri = "wss://echo.websocket.events"
    tiempos = []

    async with websockets.connect(uri) as ws:
        # Leer mensaje de bienvenida si existe
        try:
            bienvenida = await asyncio.wait_for(ws.recv(), timeout=3)
            print(f"Servidor dice: {bienvenida}")
        except asyncio.TimeoutError:
            pass

        print(f"Enviando {n} mensajes...")
        for i in range(n):
            mensaje = json.dumps({"seq": i, "ts": time.time()})
            t0 = time.perf_counter()
            await ws.send(mensaje)
            respuesta = await ws.recv()
            t1 = time.perf_counter()
            tiempos.append(t1 - t0)
            if (i + 1) % 10 == 0:
                print(f"  {i + 1}/{n} completados")

    return tiempos

# Ejecutar benchmark
tiempos_ws = await benchmark_websocket(N)
arr_ws = stats(tiempos_ws, "WebSocket RTT")

### ¿Por qué WebSocket es más rápido?

WebSocket establece la conexión **una sola vez** (upgrade desde HTTP).
Después de eso, cada mensaje:

1. **No tiene headers HTTP** — solo un frame de 2-14 bytes
2. **No tiene handshake** — la conexión ya está establecida
3. **Es bidireccional** — el servidor puede enviar sin que el cliente pregunte

Comparado con REST (incluso con sesión), cada petición HTTP incluye:
- Headers de request (~200-500 bytes)
- Headers de response (~300-800 bytes)
- Parsing de HTTP en cada intercambio

## 6. Comparación de latencia

Visualizamos las distribuciones de latencia de los tres métodos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Gráfica 1: Box plot ---
ax1 = axes[0]
data = [arr_sin_sesion, arr_con_sesion, arr_ws]
labels = ['REST\n(sin sesión)', 'REST\n(con sesión)', 'WebSocket\n(RTT)']
colores = ['#e74c3c', '#3498db', '#2ecc71']

bp = ax1.boxplot(data, labels=labels, patch_artist=True,
                 medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], colores):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax1.set_ylabel('Latencia (ms)', fontsize=12)
ax1.set_title('Distribución de latencia', fontsize=13)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Gráfica 2: Bar chart de medias ---
ax2 = axes[1]
medias = [arr_sin_sesion.mean(), arr_con_sesion.mean(), arr_ws.mean()]
p95s = [np.percentile(arr_sin_sesion, 95),
        np.percentile(arr_con_sesion, 95),
        np.percentile(arr_ws, 95)]

x = np.arange(len(labels))
width = 0.35
bars1 = ax2.bar(x - width/2, medias, width, label='Media', color=colores, alpha=0.8)
bars2 = ax2.bar(x + width/2, p95s, width, label='P95', color=colores, alpha=0.4,
                edgecolor=colores, linewidth=2)

for bar, val in zip(bars1, medias):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.0f}', ha='center', va='bottom', fontsize=10)
for bar, val in zip(bars2, p95s):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.0f}', ha='center', va='bottom', fontsize=10)

ax2.set_ylabel('Latencia (ms)', fontsize=12)
ax2.set_title('Media vs P95', fontsize=13)
ax2.set_xticks(x)
ax2.set_xticklabels(labels)
ax2.legend()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

### ¿Por qué estos resultados?

La jerarquía de latencia se explica por el **overhead de conexión**:

| Método | Conexión | Headers por mensaje | Overhead |
|--------|----------|--------------------|---------|
| REST sin sesión | Nueva cada vez (DNS + TCP + TLS) | ~500-1000 bytes | Alto |
| REST con sesión | Reutilizada (keep-alive) | ~500-1000 bytes | Medio |
| WebSocket | Una sola vez (upgrade) | ~2-14 bytes (frame) | Bajo |

## 7. Comparación de payload

Comparamos el tamaño de payload para transmitir la misma información
(un objeto con 5-6 campos) en diferentes protocolos.

Los valores de gRPC y SOAP son estimaciones basadas en la especificación de cada formato.

In [ ]:
# Tamaños de payload para un objeto equivalente
# (usuario con id, nombre, email, edad, activo)

# REST JSON
rest_payload = json.dumps({
    "id": 12345,
    "name": "Juan P\u00e9rez",
    "email": "juan@example.com",
    "age": 28,
    "active": True,
    "department": "Engineering",
    "roles": ["admin", "user"]
})
rest_size = len(rest_payload.encode('utf-8'))

# GraphQL JSON (mismos datos, estructura diferente)
graphql_payload = json.dumps({
    "data": {
        "user": {
            "name": "Juan P\u00e9rez",
            "email": "juan@example.com",
            "age": 28
        }
    }
})
graphql_size = len(graphql_payload.encode('utf-8'))

# Protobuf (estimado basado en la codificación real)
# field_tag(1) + varint(12345) = 4 bytes
# field_tag(1) + len(1) + "Juan Pérez"(12) = 14 bytes
# field_tag(1) + len(1) + "juan@example.com"(16) = 18 bytes
# field_tag(1) + varint(28) = 2 bytes
# field_tag(1) + varint(1) = 2 bytes
# field_tag(1) + len(1) + "Engineering"(11) = 13 bytes
# field_tag(1) + len(1) + "admin"(5) = 7 bytes
# field_tag(1) + len(1) + "user"(4) = 6 bytes
protobuf_size = 82  # estimación razonable

# SOAP XML
soap_payload = """<?xml version="1.0" encoding="UTF-8"?>
<soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/"
               xmlns:usr="http://example.com/users">
  <soap:Body>
    <usr:GetUserResponse>
      <usr:User>
        <usr:Id>12345</usr:Id>
        <usr:Name>Juan P\u00e9rez</usr:Name>
        <usr:Email>juan@example.com</usr:Email>
        <usr:Age>28</usr:Age>
        <usr:Active>true</usr:Active>
        <usr:Department>Engineering</usr:Department>
        <usr:Roles>
          <usr:Role>admin</usr:Role>
          <usr:Role>user</usr:Role>
        </usr:Roles>
      </usr:User>
    </usr:GetUserResponse>
  </soap:Body>
</soap:Envelope>"""
soap_size = len(soap_payload.encode('utf-8'))

print(f"Tamaño de payload (mismo contenido lógico):")
print(f"  REST (JSON completo):       {rest_size:>4} bytes")
print(f"  GraphQL (JSON parcial):     {graphql_size:>4} bytes")
print(f"  gRPC (Protobuf estimado):   {protobuf_size:>4} bytes")
print(f"  SOAP (XML):                 {soap_size:>4} bytes")

In [ ]:
# Gráfica de comparación de payload
formatos = ['SOAP\n(XML)', 'REST\n(JSON)', 'GraphQL\n(JSON)', 'gRPC\n(Protobuf)']
sizes = [soap_size, rest_size, graphql_size, protobuf_size]
colores = ['#e74c3c', '#3498db', '#9b59b6', '#2ecc71']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(formatos, sizes, color=colores, edgecolor='white', linewidth=1.5)

# Etiquetas de valor
for bar, size in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
            f'{size} B', ha='center', va='bottom', fontsize=13, fontweight='bold')

# Referencia
ax.axhline(y=protobuf_size, color='#2ecc71', linestyle='--', alpha=0.3, linewidth=1)

ax.set_ylabel('Tama\u00f1o (bytes)', fontsize=12)
ax.set_title('Comparaci\u00f3n de tama\u00f1o de payload por protocolo', fontsize=14)
ax.set_ylim(0, max(sizes) * 1.25)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

### ¿Por qué Protobuf es más pequeño?

**Protobuf** usa codificación binaria optimizada:

1. **Sin nombres de campo**: usa números (1 byte) en lugar de strings como `"department"`
2. **Sin delimitadores de texto**: no necesita `{`, `}`, `:`, `,`, `"`
3. **Tipos nativos**: un entero como `28` ocupa 1 byte (varint), no 2 bytes como en JSON (`"28"`)
4. **Sin encoding overhead**: los booleanos son 1 byte, no 4-5 bytes (`"true"`/`"false"`)

**SOAP** es el más grande porque:
1. Etiquetas XML de apertura y cierre para cada campo
2. Namespaces XML obligatorios
3. Envelope SOAP con headers
4. Declaración XML

**GraphQL** puede ser más pequeño que REST porque el cliente solo pide los campos que necesita (evita over-fetching).

## 8. Análisis: escala y volumen

¿Cómo escalan estas diferencias con el volumen de datos?

In [ ]:
# Proyección de tamaño para diferentes cantidades de registros
n_registros = [1, 10, 100, 1000, 10000]

fig, ax = plt.subplots(figsize=(10, 6))

for fmt, base_size, color, marker in [
    ('SOAP (XML)', soap_size, '#e74c3c', 'o'),
    ('REST (JSON)', rest_size, '#3498db', 's'),
    ('GraphQL (JSON)', graphql_size, '#9b59b6', '^'),
    ('gRPC (Protobuf)', protobuf_size, '#2ecc71', 'D'),
]:
    sizes_kb = [base_size * n / 1024 for n in n_registros]
    ax.plot(n_registros, sizes_kb, marker=marker, label=fmt,
            color=color, linewidth=2, markersize=8)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('N\u00famero de registros', fontsize=12)
ax.set_ylabel('Tama\u00f1o total (KB)', fontsize=12)
ax.set_title('Escalamiento del payload con el volumen de datos', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f"\nA 10,000 registros:")
for fmt, base_size in [('SOAP', soap_size), ('REST', rest_size),
                        ('GraphQL', graphql_size), ('Protobuf', protobuf_size)]:
    total_mb = base_size * 10000 / (1024 * 1024)
    print(f"  {fmt:.<20s} {total_mb:.2f} MB")

### ¿Por qué importa la escala?

A pequeña escala (1-10 registros), la diferencia entre formatos es insignificante.
Pero a escala de producción (miles o millones de mensajes):

- El **ancho de banda** se vuelve un cuello de botella
- El **tiempo de serialización/deserialización** impacta la latencia
- Los **costos de red** (cloud) se acumulan

Por eso los microservicios internos usan gRPC (eficiente en red)
mientras que las APIs públicas usan REST/GraphQL (fácil de usar).

## 9. Resumen

### Tabla de resultados

In [ ]:
print("=" * 75)
print(f"{'M\u00e9todo':<25} {'Latencia media':>15} {'P95':>12} {'Payload':>12}")
print("=" * 75)
print(f"{'REST (sin sesi\u00f3n)':<25} {arr_sin_sesion.mean():>12.1f} ms {np.percentile(arr_sin_sesion, 95):>9.1f} ms {rest_size:>9} B")
print(f"{'REST (con sesi\u00f3n)':<25} {arr_con_sesion.mean():>12.1f} ms {np.percentile(arr_con_sesion, 95):>9.1f} ms {rest_size:>9} B")
print(f"{'WebSocket':<25} {arr_ws.mean():>12.1f} ms {np.percentile(arr_ws, 95):>9.1f} ms {'~50':>9} B")
print(f"{'gRPC (estimado)':<25} {'~1-5':>15} {'~5-10':>12} {protobuf_size:>9} B")
print("-" * 75)
print(f"{'SOAP (solo payload)':<25} {'N/A':>15} {'N/A':>12} {soap_size:>9} B")
print(f"{'GraphQL (solo payload)':<25} {'N/A':>15} {'N/A':>12} {graphql_size:>9} B")
print("=" * 75)

### Conclusiones

1. **Latencia**: WebSocket < REST con sesión < REST sin sesión. La reutilización de conexiones es clave.

2. **Payload**: Protobuf < GraphQL (parcial) < REST (JSON) < SOAP (XML). La codificación binaria y la selección de campos reducen el tamaño.

3. **No hay protocolo "mejor"**:
   - REST: simple, universal, ideal para APIs públicas
   - GraphQL: flexible, evita over-fetching, ideal para frontends complejos
   - WebSocket: baja latencia, bidireccional, ideal para tiempo real
   - gRPC: eficiente, tipado, ideal para comunicación entre microservicios
   - SOAP: robusto, estandarizado, común en sistemas legacy (bancos, gobierno)

4. **La elección depende del contexto**: volumen de datos, frecuencia de comunicación, tipo de clientes, requisitos de latencia y compatibilidad con sistemas existentes.